# byteSmart Training Notebook

This notebook only prepares the data, trains the three models, and saves them as pickle files.

## 1. Setup

In [ ]:
# Keep the library list small and focused.
import pickle
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.linear_model import LinearRegression, LogisticRegression
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import (
        accuracy_score,
        classification_report,
        ConfusionMatrixDisplay,
        mean_absolute_error,
        mean_squared_error,
        r2_score,
        silhouette_score,
    )
except ImportError:
    %pip -q install scikit-learn
    from sklearn.linear_model import LinearRegression, LogisticRegression
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import (
        accuracy_score,
        classification_report,
        ConfusionMatrixDisplay,
        mean_absolute_error,
        mean_squared_error,
        r2_score,
        silhouette_score,
    )

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

## 2. Load Zip Data

In [ ]:
ZIP_NAME = '14888121-20260708T173347Z-3-001.zip'
SEARCH_ROOTS = [
    Path('/content'),
    Path('/content/drive/MyDrive'),
    Path.cwd(),
    Path('/Users/mokshjoshi/Downloads'),  # Allows local Mac verification.
]

def find_zip(zip_name=ZIP_NAME):
    for root in SEARCH_ROOTS:
        if root.exists():
            direct = root / zip_name
            if direct.exists():
                return direct
            matches = list(root.rglob(zip_name))
            if matches:
                return matches[0]
    raise FileNotFoundError(
        f'Could not find {zip_name}. Upload it into Colab or place it in Google Drive.'
    )

zip_path = find_zip()
extract_dir = Path('/content/byteSmart_data') if Path('/content').exists() else Path.cwd() / 'byteSmart_data'
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)

data_dir = extract_dir / '14888121'
print('Using data directory:', data_dir)
print('CSV files:', [p.name for p in data_dir.glob('*.csv')])

## 3. Prepare Data

In [ ]:
def elapsed_to_hours(series):
    td = pd.to_timedelta(series.astype(str), errors='coerce')
    return td.dt.total_seconds() / 3600

def load_dry_ice(path):
    raw = pd.read_csv(path, header=None)
    return pd.DataFrame({
        'hours': pd.to_numeric(raw.iloc[4:, 2], errors='coerce'),
        'baseline_lb': pd.to_numeric(raw.iloc[4:, 5], errors='coerce'),
        'refrigerated_lb': pd.to_numeric(raw.iloc[4:, 6], errors='coerce'),
    }).dropna(subset=['hours'])

def load_test1(path):
    df = pd.read_csv(path, low_memory=False).iloc[2:].copy()
    df['hours'] = elapsed_to_hours(df['Time Elapsed'])
    for col in df.columns:
        if col not in ['date', 'time', 'Time Elapsed', 'hours']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.dropna(subset=['hours'])

def load_test2(path):
    df = pd.read_csv(path, low_memory=False).iloc[1:].copy()
    df['timestamp'] = pd.to_datetime(df['TIMESTAMP'], errors='coerce')
    df['hours'] = (df['timestamp'] - df['timestamp'].min()).dt.total_seconds() / 3600
    for col in df.columns:
        if col not in ['TIMESTAMP', 'timestamp', 'hours']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.dropna(subset=['hours'])

def temperature_columns(df, exclude):
    return [
        col for col in df.columns
        if col not in exclude and pd.api.types.is_numeric_dtype(df[col]) and df[col].notna().sum() > 100
    ]

def sensor_slopes(df, cols):
    slopes = {}
    for col in cols:
        sub = df[['hours', col]].dropna()
        if len(sub) >= 2:
            slopes[col] = LinearRegression().fit(sub[['hours']], sub[col]).coef_[0]
    return pd.Series(slopes)

def make_windows(df, temp_cols, label, window_size=60):
    rows = []
    work = df[['hours', 'O2', 'CO2']].copy()
    work['mean_temp_F'] = df[temp_cols].mean(axis=1)
    work['spread_F'] = df[temp_cols].max(axis=1) - df[temp_cols].min(axis=1)
    work = work.dropna()

    for start in range(0, len(work) - window_size + 1, window_size):
        chunk = work.iloc[start:start + window_size]
        slope = LinearRegression().fit(chunk[['hours']], chunk['mean_temp_F']).coef_[0]
        rows.append({
            'label': label,
            'mean_temp_F': chunk['mean_temp_F'].mean(),
            'spread_F': chunk['spread_F'].mean(),
            'O2_mean': chunk['O2'].mean(),
            'CO2_mean': chunk['CO2'].mean(),
            'mean_temp_slope_F_per_hour': slope,
        })
    return pd.DataFrame(rows)

def sensor_feature_table(df, temp_cols, test_label):
    slopes = sensor_slopes(df, temp_cols)
    return pd.DataFrame({
        'sensor': temp_cols,
        'test': test_label,
        'mean_F': [df[c].mean() for c in temp_cols],
        'std_F': [df[c].std() for c in temp_cols],
        'min_F': [df[c].min() for c in temp_cols],
        'max_F': [df[c].max() for c in temp_cols],
        'slope_F_per_hour': [slopes.get(c, np.nan) for c in temp_cols],
    }).dropna()

dry = load_dry_ice(data_dir / 'Test1_DryIceWeight.csv')
test1 = load_test1(data_dir / 'Test1_TempCO2O2.csv')
test2 = load_test2(data_dir / 'Test2_TempCO2O2.csv')

test1_temp_cols = temperature_columns(
    test1, {'date', 'time', 'Time Elapsed', 'hours', 'O2', 'CO2', 'Ambient', 'Unnamed: 63', 'Unnamed: 64'}
)
test2_temp_cols = temperature_columns(
    test2, {'TIMESTAMP', 'timestamp', 'hours', 'O2', 'CO2'}
)

print('Dry ice rows:', dry.shape)
print('Test 1 rows and temperature sensors:', test1.shape, len(test1_temp_cols))
print('Test 2 rows and temperature sensors:', test2.shape, len(test2_temp_cols))

## 4. Build Model Datasets

In [ ]:
# Linear Regression dataset: predict dry ice mass from hours and condition.
dry_long = pd.concat([
    dry[['hours', 'baseline_lb']].rename(columns={'baseline_lb': 'mass_lb'}).assign(condition='baseline'),
    dry[['hours', 'refrigerated_lb']].rename(columns={'refrigerated_lb': 'mass_lb'}).assign(condition='refrigerated'),
], ignore_index=True).dropna()
dry_long['is_refrigerated'] = (dry_long['condition'] == 'refrigerated').astype(int)
linear_features = ['hours', 'is_refrigerated']

# Logistic Regression dataset: classify whether a time window looks like Test 1 or Test 2.
windows = pd.concat([
    make_windows(test1, test1_temp_cols, 'Test 1'),
    make_windows(test2, test2_temp_cols, 'Test 2'),
], ignore_index=True)
logistic_features = [
    'mean_temp_F',
    'spread_F',
    'O2_mean',
    'CO2_mean',
    'mean_temp_slope_F_per_hour',
]

# K-Means dataset: cluster sensor behavior based on summary statistics.
sensor_features = pd.concat([
    sensor_feature_table(test1, test1_temp_cols, 'Test 1'),
    sensor_feature_table(test2, test2_temp_cols, 'Test 2'),
], ignore_index=True)
kmeans_features = ['mean_F', 'std_F', 'min_F', 'max_F', 'slope_F_per_hour']

print('Linear Regression rows:', dry_long.shape)
print('Logistic Regression windows:', windows.shape)
print('K-Means sensor rows:', sensor_features.shape)

## 5. Train the Three Models

In [ ]:
# 1. Linear Regression train/test split and training.
X_linear = dry_long[linear_features]
y_linear = dry_long['mass_lb']
X_linear_train, X_linear_test, y_linear_train, y_linear_test = train_test_split(
    X_linear, y_linear, test_size=0.25, random_state=42
)
linear_model = LinearRegression()
linear_model.fit(X_linear_train, y_linear_train)
linear_predictions = linear_model.predict(X_linear_test)
linear_metrics = {
    'MAE': mean_absolute_error(y_linear_test, linear_predictions),
    'RMSE': mean_squared_error(y_linear_test, linear_predictions) ** 0.5,
    'R2': r2_score(y_linear_test, linear_predictions),
}
print('Linear Regression metrics:', linear_metrics)

# 2. Logistic Regression train/test split and training.
X_logistic = windows[logistic_features]
y_logistic = windows['label']
X_logistic_train, X_logistic_test, y_logistic_train, y_logistic_test = train_test_split(
    X_logistic, y_logistic, test_size=0.25, random_state=42, stratify=y_logistic
)
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_logistic_train, y_logistic_train)
logistic_predictions = logistic_model.predict(X_logistic_test)
logistic_accuracy = accuracy_score(y_logistic_test, logistic_predictions)
print('Logistic Regression accuracy:', round(logistic_accuracy, 3))
print(classification_report(y_logistic_test, logistic_predictions))

# 3. K-Means train/test split and training.
X_kmeans = sensor_features[kmeans_features]
X_kmeans_train, X_kmeans_test = train_test_split(X_kmeans, test_size=0.25, random_state=42)
kmeans_scaler = StandardScaler()
X_kmeans_train_scaled = kmeans_scaler.fit_transform(X_kmeans_train)
X_kmeans_test_scaled = kmeans_scaler.transform(X_kmeans_test)

kmeans_model = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans_model.fit(X_kmeans_train_scaled)
kmeans_test_clusters = kmeans_model.predict(X_kmeans_test_scaled)
kmeans_silhouette = silhouette_score(X_kmeans_test_scaled, kmeans_test_clusters)
print('K-Means test silhouette score:', round(kmeans_silhouette, 3))

## 6. Save the Three Pickle Models

In [ ]:
MODEL_DIR = Path('/content') if Path('/content').exists() else Path.cwd()

linear_bundle = {
    'model': linear_model,
    'features': linear_features,
    'target': 'mass_lb',
    'description': 'Predicts dry ice mass from elapsed hours and refrigeration condition.',
}
logistic_bundle = {
    'model': logistic_model,
    'features': logistic_features,
    'description': 'Classifies whether a sensor time window looks like Test 1 or Test 2.',
}
kmeans_bundle = {
    'model': kmeans_model,
    'scaler': kmeans_scaler,
    'features': kmeans_features,
    'description': 'Clusters sensors by mean, spread, min, max, and warming/cooling slope.',
}

with open(MODEL_DIR / 'linear_regression_model.pkl', 'wb') as f:
    pickle.dump(linear_bundle, f)
with open(MODEL_DIR / 'logistic_regression_model.pkl', 'wb') as f:
    pickle.dump(logistic_bundle, f)
with open(MODEL_DIR / 'kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans_bundle, f)

print('Saved model files to:', MODEL_DIR)
print('linear_regression_model.pkl')
print('logistic_regression_model.pkl')
print('kmeans_model.pkl')